In [1]:
import gcamreader
import pandas as pd
import os
from pathlib import Path
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import plotly.express as px
import matplotlib.pyplot as plt

In [2]:
# =========================================================
# Config
# =========================================================
PROJECT_PATH   = Path("../")
DB_REL_PATH    = PROJECT_PATH / "output"
DB_FILE        = "database_basexdb_korea_2035_v7"
QUERY_FILE     = DB_REL_PATH / "queries" / "Main_queries.xml"

REGION = ['South Korea']

In [3]:
# =========================================================
# DB helpers
# =========================================================
def connect_db():
    return gcamreader.LocalDBConn(DB_REL_PATH, DB_FILE)

def check_query_idx():
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    for i, q in enumerate(queries):
        print(i, q.title)

def run_query(conn, q_idx, scenarios, regions=None):
    queries = gcamreader.parse_batch_query(os.fspath(QUERY_FILE))
    q = queries[q_idx]
    print(q.title)
    df = conn.runQuery(q, scenarios=scenarios, regions=regions)
    df["scenario"] = df["scenario"].str.split(",").str[0]
    return df

def get_scenario_name(conn):
    scenarios = list(conn.listScenariosInDB()['name'].unique())
    return scenarios

In [4]:
def convert_to_mt(row):
    val, unit = row['value'], row['Units']
    if unit == 'Tg':
        return val
    elif unit == 'Gg':
        return val * 1e-3
    elif unit == 'MTC':
        return val * (44.009 / 12.011)
    else:
        raise ValueError(f"Unknown unit: {unit}")

# AR5 100‑yr GWP (no climate–carbon feedbacks)
GWP_AR5 = {
    'CO2':      1,      
    'CH4':     28,      
    'CH4_AGR': 28,
    'CH4_AWB': 28,
    'N2O':    265,      
    'N2O_AGR':265,
    'N2O_AWB':265,
    'HFC125': 3170,     
    'HFC134a':1300,     
    'HFC143a':4800,     
    'HFC23': 12400,     
    'HFC32':   677,     
    'HFC43':  1650,     
    'HFC227ea':3350,    
    'HFC236fa':8060,    
    'SF6':   23500,     
    'C2F6':  11100,     
    'CF4':    6630,     
}


In [5]:
check_query_idx()

0 primary energy consumption by region (avg fossil efficiency)
1 primary energy consumption by region (direct equivalent)
2 primary energy consumption with CCS by region (direct equivalent)
3 resource production
4 resource production by tech and vintage
5 resource supply curves
6 regional primary energy prices
7 elec gen by region (incl CHP)
8 elec gen by subsector
9 elec gen by gen tech
10 elec gen by gen tech and cooling tech
11 elec gen by gen tech and cooling tech and vintage
12 elec gen by gen tech and cooling tech (new)
13 elec energy input by subsector
14 elec energy input by elec gen tech
15 elec energy input by elec gen tech and cooling tech
16 elec prices by sector
17 elec gen costs by subsector
18 elec gen costs by tech
19 elec gen costs by cooling tech
20 elec share-weights by subsector
21 elec share-weights by tech
22 elec share-weights by cooling tech
23 elec td inputs and outputs
24 cogeneration by region
25 elec consumption by demand sector
26 elec sector water withdraw

In [6]:
conn = connect_db()
get_scenario_name(conn)

Database scenarios: High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, High-Ambition-Med, Current-Policies-Med, High-Ambition-High, High-Ambition-Low, High-Ambition-Med-AI, High-Ambition-Med-CPO2040, Current-Policies-Low, Current-Policies-High, Current-Policies-Med-AI


['High-Ambition-Med',
 'Current-Policies-Med',
 'High-Ambition-High',
 'High-Ambition-Low',
 'High-Ambition-Med-AI',
 'High-Ambition-Med-CPO2040',
 'Current-Policies-Low',
 'Current-Policies-High',
 'Current-Policies-Med-AI']

In [18]:
scenarios = [
    'Current-Policies-Med',
    'Current-Policies-Med-AI',
    'High-Ambition-Med',
    'High-Ambition-Med-AI'
]

In [28]:
df = run_query(conn=conn, q_idx=16, scenarios=scenarios, regions=['South Korea'])
df[(df['Year'] == 2035) & (df['fuel']=='elect_td_ind')]

elec prices by sector


,Units,scenario,region,fuel,Year,value
30,1975$/GJ,Current-Policies-Med,South Korea,elect_td_ind,2035,9.86819
118,1975$/GJ,Current-Policies-Med-AI,South Korea,elect_td_ind,2035,10.87510
206,1975$/GJ,High-Ambition-Med,South Korea,elect_td_ind,2035,13.53560
294,1975$/GJ,High-Ambition-Med-AI,South Korea,elect_td_ind,2035,14.75420


In [29]:
10.87510 / 9.86819

1.1020359356680405

In [31]:
 14.75420 / 13.53560

1.0900292561836933

In [27]:
12.05080 / 11.93040

1.0100918661570442

In [22]:
df = run_query(conn=conn, q_idx=9, scenarios=scenarios, regions=['South Korea'])
df[(df['Year'] >= 2030) & (df['Year'] <= 2035)].groupby(['Year','scenario'])['value'].sum() * 277.8

elec gen by gen tech


Year  scenario               
2030  Current-Policies-Med       655.343698
      Current-Policies-Med-AI    673.816868
      High-Ambition-Med          688.690838
      High-Ambition-Med-AI       705.615536
2035  Current-Policies-Med       690.887656
      Current-Policies-Med-AI    710.444592
      High-Ambition-Med          795.148776
      High-Ambition-Med-AI       807.514574
Name: value, dtype: float64

In [7]:
df = run_query(conn=conn, q_idx=6, scenarios=['High-Ambition-Med',
 'Current-Policies-Med',
 'High-Ambition-High',
 'High-Ambition-Low',
 'High-Ambition-Med-AI',
 'High-Ambition-Med-CPO2040',
 'Current-Policies-Low',
 'Current-Policies-High',
 'Current-Policies-Med-AI'], regions=['South Korea'])
df

regional primary energy prices


,Units,scenario,region,fuel,Year,value
0,1975$/GJ,Current-Policies-High,South Korea,nuclearFuelGenIII,1975,0.171983
1,1975$/GJ,Current-Policies-High,South Korea,nuclearFuelGenIII,1990,0.193413
2,1975$/GJ,Current-Policies-High,South Korea,nuclearFuelGenIII,2005,0.222914
3,1975$/GJ,Current-Policies-High,South Korea,nuclearFuelGenIII,2010,0.232708
4,1975$/GJ,Current-Policies-High,South Korea,nuclearFuelGenIII,2015,0.241887
...,...,...,...,...,...,...
985,1975$/GJ,High-Ambition-Med-CPO2040,South Korea,regional oil,2080,0.000000
986,1975$/GJ,High-Ambition-Med-CPO2040,South Korea,regional oil,2085,0.000000
987,1975$/GJ,High-Ambition-Med-CPO2040,South Korea,regional oil,2090,0.000000
988,1975$/GJ,High-Ambition-Med-CPO2040,South Korea,regional oil,2095,0.000000


In [8]:
df['fuel'].unique()

array(['nuclearFuelGenIII', 'regional biomass', 'regional coal',
       'regional natural gas', 'regional oil'], dtype=object)

In [15]:
(2.79898 / 3.45809) ** (1/10)

0.9790758770152328

In [16]:
(5.68520 / 3.45809) ** (1/10)

1.0509715155961978

In [17]:
(3.48840 / 3.45809) ** (1/10)

1.0008730573772446

In [14]:
df[(df['fuel']=='regional natural gas') & (df['Year']>=2025) & (df['Year'] <= 2035)]

,Units,scenario,region,fuel,Year,value
72,1975$/GJ,Current-Policies-High,South Korea,regional natural gas,2025,3.45809
73,1975$/GJ,Current-Policies-High,South Korea,regional natural gas,2030,3.12009
74,1975$/GJ,Current-Policies-High,South Korea,regional natural gas,2035,2.79898
182,1975$/GJ,Current-Policies-Low,South Korea,regional natural gas,2025,3.45809
183,1975$/GJ,Current-Policies-Low,South Korea,regional natural gas,2030,4.56258
184,1975$/GJ,Current-Policies-Low,South Korea,regional natural gas,2035,5.68520
292,1975$/GJ,Current-Policies-Med,South Korea,regional natural gas,2025,3.45809
293,1975$/GJ,Current-Policies-Med,South Korea,regional natural gas,2030,3.46900
294,1975$/GJ,Current-Policies-Med,South Korea,regional natural gas,2035,3.48840
402,1975$/GJ,Current-Policies-Med-AI,South Korea,regional natural gas,2025,3.45809


In [115]:
df = run_query(conn=conn, q_idx=118, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

iron and steel production by region


,Units,scenario,region,sector,Year,value
0,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,2030,2.837300
1,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,2035,10.024080
2,Mt,High-Ambition-Med,South Korea,iron and steel,1975,1.163048
3,Mt,High-Ambition-Med,South Korea,iron and steel,1990,23.124960
4,Mt,High-Ambition-Med,South Korea,iron and steel,2005,47.820000
5,Mt,High-Ambition-Med,South Korea,iron and steel,2010,58.913960
6,Mt,High-Ambition-Med,South Korea,iron and steel,2015,69.669959
7,Mt,High-Ambition-Med,South Korea,iron and steel,2020,69.619573
8,Mt,High-Ambition-Med,South Korea,iron and steel,2025,59.372152
9,Mt,High-Ambition-Med,South Korea,iron and steel,2030,56.447383


In [116]:
df = run_query(conn=conn, q_idx=123, scenarios=['High-Ambition-Med'], regions=['USA'])
df

traded iron and steel


,Units,scenario,region,sector,subsector,input,Year,value
0,Mt,High-Ambition-Med,USA,traded iron and steel,Africa_Eastern traded iron and steel,iron and steel,1990,0.008201
1,Mt,High-Ambition-Med,USA,traded iron and steel,Africa_Eastern traded iron and steel,iron and steel,2010,0.000518
2,Mt,High-Ambition-Med,USA,traded iron and steel,Africa_Eastern traded iron and steel,iron and steel,2015,0.000238
3,Mt,High-Ambition-Med,USA,traded iron and steel,Africa_Eastern traded iron and steel,iron and steel,2020,0.532342
4,Mt,High-Ambition-Med,USA,traded iron and steel,Africa_Eastern traded iron and steel,iron and steel,2025,1.575790
...,...,...,...,...,...,...,...,...
264,Mt,High-Ambition-Med,USA,traded iron and steel,USA traded iron and steel,iron and steel,2015,10.000000
265,Mt,High-Ambition-Med,USA,traded iron and steel,USA traded iron and steel,iron and steel,2020,8.896600
266,Mt,High-Ambition-Med,USA,traded iron and steel,USA traded iron and steel,iron and steel,2025,15.825400
267,Mt,High-Ambition-Med,USA,traded iron and steel,USA traded iron and steel,iron and steel,2030,19.231400


In [117]:
df[(df['Year'].isin([2015, 2020, 2025, 2030, 2035]) & (df['subsector'] == 'South Korea traded iron and steel'))]

,Units,scenario,region,sector,subsector,input,Year,value
232,Mt,High-Ambition-Med,USA,traded iron and steel,South Korea traded iron and steel,Export-Floor,2020,28.8997
237,Mt,High-Ambition-Med,USA,traded iron and steel,South Korea traded iron and steel,iron and steel,2015,31.1730
238,Mt,High-Ambition-Med,USA,traded iron and steel,South Korea traded iron and steel,iron and steel,2020,28.8997
239,Mt,High-Ambition-Med,USA,traded iron and steel,South Korea traded iron and steel,iron and steel,2025,24.0550
240,Mt,High-Ambition-Med,USA,traded iron and steel,South Korea traded iron and steel,iron and steel,2030,23.7695
241,Mt,High-Ambition-Med,USA,traded iron and steel,South Korea traded iron and steel,iron and steel,2035,24.7120


In [120]:
df = run_query(conn=conn, q_idx=120, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df[(df['subsector'] == 'BLASTFUR')]

iron and steel production by tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
3,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2025,0.344127
4,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2030,0.281343
5,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2025",iron and steel,2035,0.172057
6,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2030",iron and steel,2030,1.383150
7,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR CCS,year=2030",iron and steel,2035,1.130360
8,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2025",iron and steel,2025,0.126984
9,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2025",iron and steel,2030,0.103812
10,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2025",iron and steel,2035,0.063482
11,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2030",iron and steel,2030,0.346517
12,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,"BLASTFUR with hydrogen,year=2030",iron and steel,2035,0.283006


In [86]:
df = run_query(conn=conn, q_idx=122, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

iron and steel prices


,Units,scenario,region,sector,Year,value
0,1975$/kg,High-Ambition-Med,South Korea,iron and steel,1975,0.125271
1,1975$/kg,High-Ambition-Med,South Korea,iron and steel,1990,0.131144
2,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2005,0.140742
3,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2010,0.139934
4,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2015,0.143461
5,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2020,0.163952
6,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2025,0.164863
7,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2030,0.184781
8,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2035,0.193700
9,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2040,0.000000


In [87]:
df = run_query(conn=conn, q_idx=316, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

demand of all markets


,Units,scenario,Year,market,value
0,EJ,High-Ambition-Med,1975,South KoreaCO2 removal-tfe,0.0
1,EJ,High-Ambition-Med,1975,South KoreaH2 central production,0.0
2,EJ,High-Ambition-Med,1975,South KoreaH2 industrial,0.0
3,EJ,High-Ambition-Med,1975,South KoreaH2 liquid truck,0.0
4,EJ,High-Ambition-Med,1975,South KoreaH2 pipeline,0.0
...,...,...,...,...,...
8245,unitless,High-Ambition-Med,2090,South KoreaFoodDemand_Staples-budget-fraction-...,0.0
8246,unitless,High-Ambition-Med,2095,South KoreaFoodDemand_NonStaples-budget-fracti...,0.0
8247,unitless,High-Ambition-Med,2095,South KoreaFoodDemand_Staples-budget-fraction-...,0.0
8248,unitless,High-Ambition-Med,2100,South KoreaFoodDemand_NonStaples-budget-fracti...,0.0


In [88]:
df[(df['market'].str.contains('Scrap'))]

,Units,scenario,Year,market,value
4389,EJ_or_Share,High-Ambition-Med,1975,South KoreaScrap-Floor,0.0
4405,EJ_or_Share,High-Ambition-Med,1990,South KoreaScrap-Floor,0.0
4421,EJ_or_Share,High-Ambition-Med,2005,South KoreaScrap-Floor,0.0
4437,EJ_or_Share,High-Ambition-Med,2010,South KoreaScrap-Floor,0.0
4453,EJ_or_Share,High-Ambition-Med,2015,South KoreaScrap-Floor,0.0
4469,EJ_or_Share,High-Ambition-Med,2020,South KoreaScrap-Floor,0.0
4485,EJ_or_Share,High-Ambition-Med,2025,South KoreaScrap-Floor,0.0
4501,EJ_or_Share,High-Ambition-Med,2030,South KoreaScrap-Floor,10.9
4517,EJ_or_Share,High-Ambition-Med,2035,South KoreaScrap-Floor,26.4
4533,EJ_or_Share,High-Ambition-Med,2040,South KoreaScrap-Floor,0.0


In [89]:
df = run_query(conn=conn, q_idx=315, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

supply of all markets


,Units,scenario,Year,market,value
0,EJ,High-Ambition-Med,1975,South KoreaCO2 removal-tfe,0.0
1,EJ,High-Ambition-Med,1975,South KoreaH2 central production,0.0
2,EJ,High-Ambition-Med,1975,South KoreaH2 industrial,0.0
3,EJ,High-Ambition-Med,1975,South KoreaH2 liquid truck,0.0
4,EJ,High-Ambition-Med,1975,South KoreaH2 pipeline,0.0
...,...,...,...,...,...
8245,unitless,High-Ambition-Med,2090,South KoreaFoodDemand_Staples-budget-fraction-...,0.0
8246,unitless,High-Ambition-Med,2095,South KoreaFoodDemand_NonStaples-budget-fracti...,0.0
8247,unitless,High-Ambition-Med,2095,South KoreaFoodDemand_Staples-budget-fraction-...,0.0
8248,unitless,High-Ambition-Med,2100,South KoreaFoodDemand_NonStaples-budget-fracti...,0.0


In [90]:
df[(df['market'].str.contains('Scrap'))]

,Units,scenario,Year,market,value
4389,EJ_or_Share,High-Ambition-Med,1975,South KoreaScrap-Floor,0.000
4405,EJ_or_Share,High-Ambition-Med,1990,South KoreaScrap-Floor,0.000
4421,EJ_or_Share,High-Ambition-Med,2005,South KoreaScrap-Floor,0.000
4437,EJ_or_Share,High-Ambition-Med,2010,South KoreaScrap-Floor,0.000
4453,EJ_or_Share,High-Ambition-Med,2015,South KoreaScrap-Floor,0.000
4469,EJ_or_Share,High-Ambition-Med,2020,South KoreaScrap-Floor,0.000
4485,EJ_or_Share,High-Ambition-Med,2025,South KoreaScrap-Floor,0.000
4501,EJ_or_Share,High-Ambition-Med,2030,South KoreaScrap-Floor,11.755
4517,EJ_or_Share,High-Ambition-Med,2035,South KoreaScrap-Floor,26.400
4533,EJ_or_Share,High-Ambition-Med,2040,South KoreaScrap-Floor,0.000


In [91]:
df = run_query(conn=conn, q_idx=285, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

GDP per capita MER by region


,Units,scenario,region,Year,account,value
0,thous 1990$ percap,High-Ambition-Med,South Korea,1975,gdp-per-capita,1.86828
1,thous 1990$ percap,High-Ambition-Med,South Korea,1990,gdp-per-capita,5.57778
2,thous 1990$ percap,High-Ambition-Med,South Korea,2005,gdp-per-capita,12.11920
3,thous 1990$ percap,High-Ambition-Med,South Korea,2010,gdp-per-capita,14.57260
4,thous 1990$ percap,High-Ambition-Med,South Korea,2015,gdp-per-capita,16.46860
5,thous 1990$ percap,High-Ambition-Med,South Korea,2020,gdp-per-capita,18.73740
6,thous 1990$ percap,High-Ambition-Med,South Korea,2025,gdp-per-capita,21.42870
7,thous 1990$ percap,High-Ambition-Med,South Korea,2030,gdp-per-capita,22.97130
8,thous 1990$ percap,High-Ambition-Med,South Korea,2035,gdp-per-capita,23.66880
9,thous 1990$ percap,High-Ambition-Med,South Korea,2040,gdp-per-capita,29.79240


In [92]:
21.42870 / 18.73740

1.1436325210541483

In [93]:
df = run_query(conn=conn, q_idx=124, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

regional iron and steel sources


,Units,scenario,region,sector,subsector,input,Year,value
0,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,1975,1.06300
1,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,1990,15.55300
2,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2005,31.69600
3,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2010,34.28600
4,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2015,38.49700
5,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2020,40.71990
6,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2025,35.31720
7,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2030,32.66770
8,Mt,High-Ambition-Med,South Korea,regional iron and steel,domestic iron and steel,iron and steel,2035,30.26830
9,Mt,High-Ambition-Med,South Korea,regional iron and steel,imported iron and steel,Import-Ceiling,2020,12.40000


In [94]:
df.groupby(['Year'])['value'].sum()

Year
1975     2.90469
1990    21.38199
2005    50.90930
2010    59.66820
2015    61.34540
2020    65.51990
2025    55.42130
2030    54.61070
2035    53.33280
Name: value, dtype: float64

In [95]:
49 / 55.8

0.878136200716846

In [96]:
df = run_query(conn=conn, q_idx=118, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

iron and steel production by region


,Units,scenario,region,sector,Year,value
0,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,2030,2.832490
1,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,2035,10.020920
2,Mt,High-Ambition-Med,South Korea,iron and steel,1975,1.163048
3,Mt,High-Ambition-Med,South Korea,iron and steel,1990,23.124960
4,Mt,High-Ambition-Med,South Korea,iron and steel,2005,47.820000
5,Mt,High-Ambition-Med,South Korea,iron and steel,2010,58.913960
6,Mt,High-Ambition-Med,South Korea,iron and steel,2015,69.669959
7,Mt,High-Ambition-Med,South Korea,iron and steel,2020,69.619573
8,Mt,High-Ambition-Med,South Korea,iron and steel,2025,59.372152
9,Mt,High-Ambition-Med,South Korea,iron and steel,2030,56.404005


In [97]:
df = run_query(conn=conn, q_idx=119, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

iron and steel production by tech


,Units,scenario,region,sector,subsector,technology,output,Year,value
0,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,EAF with DRI,Hydrogen-based DRI,Ironsteel_early_deployment,2030,2.832490
1,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,EAF with DRI,Hydrogen-based DRI,Ironsteel_early_deployment,2035,10.020920
2,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,1975,0.665514
3,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,1990,13.226000
4,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2005,26.767200
5,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2010,34.108100
6,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2015,48.479100
7,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2020,47.580840
8,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2025,39.010770
9,Mt,High-Ambition-Med,South Korea,iron and steel,BLASTFUR,BLASTFUR,iron and steel,2030,27.965480


In [61]:
18.545990 * 1.15

21.327888499999997

In [63]:
18.545990 * 1.2 ** 2

26.7062256

In [29]:
49 / 55.8

0.878136200716846

In [71]:
17.440930 * 1.3

22.673209000000003

In [72]:
17.440930 * 1.69

29.4751717

In [98]:
df = run_query(conn=conn, q_idx=120, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df[(df['Year'] >= 2030) & (df['subsector'] == 'EAF with scrap')]

iron and steel production by tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
69,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2015",iron and steel,2030,3.805850
72,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2020",iron and steel,2030,2.093760
73,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2020",iron and steel,2035,0.763879
75,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2030,3.835540
76,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2025",iron and steel,2035,2.345590
77,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2030,11.755000
78,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2030",iron and steel,2035,9.610480
79,Mt,High-Ambition-Med,South Korea,iron and steel,EAF with scrap,"EAF with scrap,year=2035",iron and steel,2035,16.789500


In [77]:
29.5 - (0.763912 + 2.345680)

26.390408

In [75]:
22.7 * (9.608990 / 11.754900)

18.55601264153672

In [74]:
22.7 - 11.8

10.899999999999999

In [23]:
8.852590 + 3.546240

12.39883

In [69]:
df = run_query(conn=conn, q_idx=122, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

iron and steel prices


,Units,scenario,region,sector,Year,value
0,1975$/kg,High-Ambition-Med,South Korea,iron and steel,1975,0.125271
1,1975$/kg,High-Ambition-Med,South Korea,iron and steel,1990,0.131144
2,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2005,0.140742
3,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2010,0.139934
4,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2015,0.143461
5,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2020,0.163952
6,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2025,0.164863
7,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2030,0.184781
8,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2035,0.216900
9,1975$/kg,High-Ambition-Med,South Korea,iron and steel,2040,0.000000


In [99]:
0.164863 * 0.13 * 0.5

0.010716095

In [100]:
df = run_query(conn=conn, q_idx=118, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df

iron and steel production by region


,Units,scenario,region,sector,Year,value
0,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,2030,2.837300
1,EJ_or_Share,High-Ambition-Med,South Korea,iron and steel,2035,10.024080
2,Mt,High-Ambition-Med,South Korea,iron and steel,1975,1.163048
3,Mt,High-Ambition-Med,South Korea,iron and steel,1990,23.124960
4,Mt,High-Ambition-Med,South Korea,iron and steel,2005,47.820000
5,Mt,High-Ambition-Med,South Korea,iron and steel,2010,58.913960
6,Mt,High-Ambition-Med,South Korea,iron and steel,2015,69.669959
7,Mt,High-Ambition-Med,South Korea,iron and steel,2020,69.619573
8,Mt,High-Ambition-Med,South Korea,iron and steel,2025,59.372152
9,Mt,High-Ambition-Med,South Korea,iron and steel,2030,56.447383


In [12]:
0.233415 / 0.003960

58.94318181818182

In [48]:
df.groupby(['Year'])['value'].sum() * 277.8

Year
1990     99.491837
2005    364.976728
2010    471.223156
2015    521.782489
2020    537.796661
2025    593.245470
2030    695.428622
2035    808.801446
Name: value, dtype: float64

In [38]:
df['sector'].unique()

array(['CO2 removal', 'H2 central production', 'H2 industrial',
       'H2 liquid truck', 'H2 pipeline', 'H2 wholesale dispensing',
       'agricultural energy use', 'cement', 'chemical energy use',
       'comm cooling', 'comm heating', 'comm others',
       'construction energy use', 'desalinated water', 'food processing',
       'industrial wastewater treatment', 'industrial water abstraction',
       'industrial water treatment', 'iron and steel',
       'irrigation water abstraction', 'mining energy use',
       'municipal wastewater treatment', 'municipal water abstraction',
       'municipal water distribution', 'municipal water treatment',
       'other industrial energy use', 'paper',
       'process heat food processing', 'process heat paper', 'refining',
       'resid cooling modern_d1', 'resid cooling modern_d10',
       'resid cooling modern_d2', 'resid cooling modern_d3',
       'resid cooling modern_d4', 'resid cooling modern_d5',
       'resid cooling modern_d6', 'resid

In [8]:
df = run_query(conn=conn, q_idx=25, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df.groupby(['scenario', 'Year'])['value'].sum()

elec consumption by demand sector


scenario           Year
High-Ambition-Med  1975    0.039219
                   1990    0.339822
                   2005    1.278552
                   2010    1.650040
                   2015    1.816829
                   2020    1.873412
                   2025    2.065126
                   2030    2.404675
                   2035    2.845828
Name: value, dtype: float64

In [9]:
df['sector'].unique()

array(['CO2 removal', 'H2 central production', 'ammonia', 'cement',
       'chemical energy use', 'chemical feedstocks',
       'construction feedstocks', 'elec_coal (IGCC CCS)',
       'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)', 'iron and steel',
       'other industrial feedstocks', 'process heat dac', 'refining',
       'waste biomass for paper'], dtype=object)

In [18]:
df[(~df['sector'].str.contains('feedstocks')) & (~df['sector'].isin(['CO2 removal']))].groupby(['Year'])['value'].sum() * (44/12)

Year
2020     0.196288
2025     4.279787
2030    11.773688
2035    21.350798
Name: value, dtype: float64

In [42]:
df = run_query(conn=conn, q_idx=159, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df#['sector'].unique()#.groupby(['Year'])['value'].sum()

transport service output by tech


,Units,scenario,region,sector,subsector,technology,Year,value
0,million pass-km,High-Ambition-Med,South Korea,trn_aviation_intl,International Aviation,BEV,2035,1.347100e-03
1,million pass-km,High-Ambition-Med,South Korea,trn_aviation_intl,International Aviation,Hydrogen,2035,1.027110e-03
2,million pass-km,High-Ambition-Med,South Korea,trn_aviation_intl,International Aviation,Liquids,1975,6.166440e+02
3,million pass-km,High-Ambition-Med,South Korea,trn_aviation_intl,International Aviation,Liquids,1990,4.066310e+03
4,million pass-km,High-Ambition-Med,South Korea,trn_aviation_intl,International Aviation,Liquids,2005,6.792510e+03
...,...,...,...,...,...,...,...,...
270,million ton-km,High-Ambition-Med,South Korea,trn_shipping_intl,International Ship,Liquids,2015,3.207670e+06
271,million ton-km,High-Ambition-Med,South Korea,trn_shipping_intl,International Ship,Liquids,2020,3.599340e+06
272,million ton-km,High-Ambition-Med,South Korea,trn_shipping_intl,International Ship,Liquids,2025,3.210820e+06
273,million ton-km,High-Ambition-Med,South Korea,trn_shipping_intl,International Ship,Liquids,2030,2.079390e+06


In [43]:
df[(df['subsector'] == 'Car')]

,Units,scenario,region,sector,subsector,technology,Year,value
133,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,BEV,2020,1622.2300
134,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,BEV,2025,11134.1500
135,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,BEV,2030,91786.7550
136,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,BEV,2035,135190.0350
137,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,FCEV,2020,145.0600
138,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,FCEV,2025,610.9400
139,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,FCEV,2030,11662.7040
140,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,FCEV,2035,13393.9021
141,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,Hybrid Liquids,2020,5624.2800
142,million pass-km,High-Ambition-Med,South Korea,trn_pass_road_LDV_4W,Car,Hybrid Liquids,2025,30938.2300


In [34]:
df[(df['sector'].isin(['elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)']))].groupby(['Year'])['value'].sum() * 3.666667

Year
2025     2.555980
2030    13.947371
2035    43.742494
Name: value, dtype: float64

In [29]:
df[(df['sector'].isin(['elec_coal (IGCC CCS)','elec_coal (conv pul CCS)', 'elec_gas (CC CCS)']))].groupby(['Year'])['value'].sum()

Year
2025     0.898210
2030     4.202430
2035    13.076483
Name: value, dtype: float64

In [31]:
df[(df['technology'].str.contains('CCS'))].groupby(['Year'])['value'].sum()

Year
2020     0.053533
2025     1.167215
2030     5.213813
2035    14.460385
Name: value, dtype: float64

In [33]:
14.5 * 3.666667

53.1666715

In [25]:
0.203046 * 5/11.929770 

0.08510055097457873

In [10]:
df = run_query(conn=conn, q_idx=11, scenarios=['Current-Policies-Med'], regions=['South Korea'])
df[(df['Year'] >= 2025) & (df['subsector'] == 'refined liquids')]

elec gen by gen tech and cooling tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
269,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2020",elec_refined liquids (CC),2025,0.000186
270,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2020",elec_refined liquids (CC),2030,0.000131
271,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2020",elec_refined liquids (CC),2035,0.000108
272,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2025",elec_refined liquids (CC),2025,0.000622
273,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2025",elec_refined liquids (CC),2030,0.000246
274,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (dry cooling),year=2025",elec_refined liquids (CC),2035,0.000204
276,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2020",elec_refined liquids (CC),2025,0.007017
277,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2020",elec_refined liquids (CC),2030,0.005164
278,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2020",elec_refined liquids (CC),2035,0.004343
279,EJ,Current-Policies-Med,South Korea,electricity,refined liquids,"refined liquids (CC) (recirculating),year=2025",elec_refined liquids (CC),2025,0.020147


In [8]:
df = run_query(conn=conn, q_idx=267, scenarios=['High-Ambition-Med'], regions=['South Korea'])
df#[(df['Year'] >= 2025) & (df['Year'] == 2035)]

CO2 sequestration by sector


,Units,scenario,region,sector,Year,value
0,MTC,High-Ambition-Med,South Korea,CO2 removal,2025,4.823451e-07
1,MTC,High-Ambition-Med,South Korea,CO2 removal,2030,8.072014e-01
2,MTC,High-Ambition-Med,South Korea,CO2 removal,2035,1.614500e+00
3,MTC,High-Ambition-Med,South Korea,H2 central production,2025,9.155242e-03
4,MTC,High-Ambition-Med,South Korea,H2 central production,2030,7.538695e-04
...,...,...,...,...,...,...
59,MTC,High-Ambition-Med,South Korea,refining,2035,3.559288e-02
60,MTC,High-Ambition-Med,South Korea,waste biomass for paper,2020,5.433540e-05
61,MTC,High-Ambition-Med,South Korea,waste biomass for paper,2025,4.046484e-04
62,MTC,High-Ambition-Med,South Korea,waste biomass for paper,2030,3.516565e-03


In [9]:
df['sector'].unique()

array(['CO2 removal', 'H2 central production', 'ammonia', 'cement',
       'chemical energy use', 'chemical feedstocks',
       'construction feedstocks', 'elec_coal (IGCC CCS)',
       'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)', 'iron and steel',
       'other industrial feedstocks', 'process heat dac', 'refining',
       'waste biomass for paper'], dtype=object)

In [10]:
df[(df['Year'] == 2035) & (df['sector'].isin(['elec_coal (IGCC CCS)',
       'elec_coal (conv pul CCS)', 'elec_gas (CC CCS)']))].groupby(['Year'])['value'].sum() * 3.66667

Year
2035    16.206605
Name: value, dtype: float64

In [ ]:
df = run_query(conn=conn, q_idx=285, scenarios=['Current-Policies-Med'], regions=['South Korea'])
df#[(df['Year'] >= 2025) & (df['Year'] == 2035)]

GDP per capita MER by region


,Units,scenario,region,Year,account,value
0,thous 1990$ percap,Current-Policies-Med,South Korea,1975,gdp-per-capita,1.86828
1,thous 1990$ percap,Current-Policies-Med,South Korea,1990,gdp-per-capita,5.57778
2,thous 1990$ percap,Current-Policies-Med,South Korea,2005,gdp-per-capita,12.11920
3,thous 1990$ percap,Current-Policies-Med,South Korea,2010,gdp-per-capita,14.57260
4,thous 1990$ percap,Current-Policies-Med,South Korea,2015,gdp-per-capita,16.46860
5,thous 1990$ percap,Current-Policies-Med,South Korea,2020,gdp-per-capita,18.73740
6,thous 1990$ percap,Current-Policies-Med,South Korea,2025,gdp-per-capita,21.42870
7,thous 1990$ percap,Current-Policies-Med,South Korea,2030,gdp-per-capita,22.97130
8,thous 1990$ percap,Current-Policies-Med,South Korea,2035,gdp-per-capita,23.66880
9,thous 1990$ percap,Current-Policies-Med,South Korea,2040,gdp-per-capita,29.79240


In [8]:
df = run_query(conn=conn, q_idx=285, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df#[(df['Year'] >= 2025) & (df['Year'] == 2035)]

GDP per capita MER by region


,Units,scenario,region,Year,account,value
0,thous 1990$ percap,Other-EA,South Korea,1975,gdp-per-capita,1.76758
1,thous 1990$ percap,Other-EA,South Korea,1990,gdp-per-capita,5.69465
2,thous 1990$ percap,Other-EA,South Korea,2005,gdp-per-capita,12.88740
3,thous 1990$ percap,Other-EA,South Korea,2010,gdp-per-capita,15.47690
4,thous 1990$ percap,Other-EA,South Korea,2015,gdp-per-capita,17.47210
5,thous 1990$ percap,Other-EA,South Korea,2020,gdp-per-capita,19.07780
6,thous 1990$ percap,Other-EA,South Korea,2025,gdp-per-capita,21.78420
7,thous 1990$ percap,Other-EA,South Korea,2030,gdp-per-capita,23.92870
8,thous 1990$ percap,Other-EA,South Korea,2035,gdp-per-capita,25.93040
9,thous 1990$ percap,Other-EA,South Korea,2040,gdp-per-capita,27.82850


In [14]:
df = run_query(conn=conn, q_idx=283, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df#[(df['Year'] >= 2025) & (df['Year'] == 2035)]

population by region


,Units,scenario,region,Year,value
0,thous,Other-EA,South Korea,1975,35281
1,thous,Other-EA,South Korea,1990,42869
2,thous,Other-EA,South Korea,2005,48185
3,thous,Other-EA,South Korea,2010,49554
4,thous,Other-EA,South Korea,2015,51015
5,thous,Other-EA,South Korea,2020,51836
6,thous,Other-EA,South Korea,2025,51448
7,thous,Other-EA,South Korea,2030,51199
8,thous,Other-EA,South Korea,2035,50869
9,thous,Other-EA,South Korea,2040,50193


In [4]:
21.78420 * 51448

1120753.5215999999

In [5]:
23.35239 * 51305

1198094.36895

In [6]:
23.82319 * 50824

1210789.80856

In [7]:
22.85046 * 50690

1158289.8174

In [8]:
22.87071 * 49470

1131414.0237

In [9]:
23.81664 * 51949

1237250.63136

In [10]:
25.53114 * 52209

1332955.28826

In [11]:
np.exp(np.log(23.92870 / 21.78420) / 5)

1.0189561569900778

In [13]:
np.exp(np.log(25.93040 / 23.92870) / 5)

1.0161972662959566

In [24]:
df = run_query(conn=conn, q_idx=70, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df[(df['Year'] >= 2025) & (df['Year'] == 2035)]

building floorspace


,Units,scenario,region,building,nodeInput,building-node-input,Year,value
8,billion m^2,Other-CP,South Korea,comm,comm,comm_building,2035,0.845120
30,billion m^2,Other-CP,South Korea,resid_d1,resid,resid_building,2035,0.110299
52,billion m^2,Other-CP,South Korea,resid_d10,resid,resid_building,2035,0.159799
74,billion m^2,Other-CP,South Korea,resid_d2,resid,resid_building,2035,0.119527
96,billion m^2,Other-CP,South Korea,resid_d3,resid,resid_building,2035,0.123280
118,billion m^2,Other-CP,South Korea,resid_d4,resid,resid_building,2035,0.125949
140,billion m^2,Other-CP,South Korea,resid_d5,resid,resid_building,2035,0.128202
162,billion m^2,Other-CP,South Korea,resid_d6,resid,resid_building,2035,0.130435
184,billion m^2,Other-CP,South Korea,resid_d7,resid,resid_building,2035,0.132969
206,billion m^2,Other-CP,South Korea,resid_d8,resid,resid_building,2035,0.136192


In [25]:
df = run_query(conn=conn, q_idx=95, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df[(df['Year'] >= 2025) & (df['Year'] <= 2030)]

building share-weights by tech


,Units,scenario,region,sector,subsector,Year,technology,value
6,None Specified,Other-CP,South Korea,comm cooling,electricity,2025,electricity,1.000000
7,None Specified,Other-CP,South Korea,comm cooling,electricity,2030,electricity,1.000000
40,None Specified,Other-CP,South Korea,comm cooling,gas,2025,gas,1.000000
41,None Specified,Other-CP,South Korea,comm cooling,gas,2030,gas,1.000000
74,None Specified,Other-CP,South Korea,comm heating,biomass,2025,biomass,1.000000
...,...,...,...,...,...,...,...,...
9251,None Specified,Other-EA,South Korea,resid others modern_d9,gas,2025,hydrogen,0.166667
9252,None Specified,Other-EA,South Korea,resid others modern_d9,gas,2030,gas,0.500000
9253,None Specified,Other-EA,South Korea,resid others modern_d9,gas,2030,hydrogen,0.333333
9301,None Specified,Other-EA,South Korea,resid others modern_d9,refined liquids,2025,refined liquids,1.000000


In [27]:
df[(df['sector'].str.contains('resid')) & (df['technology'] == 'gas') & (df['Year'] >= 2015) & (df['Year'] ==2030)]

,Units,scenario,region,sector,subsector,Year,technology,value
426,None Specified,Other-CP,South Korea,resid cooling modern_d1,gas,2030,gas,1.0
494,None Specified,Other-CP,South Korea,resid cooling modern_d10,gas,2030,gas,1.0
562,None Specified,Other-CP,South Korea,resid cooling modern_d2,gas,2030,gas,1.0
630,None Specified,Other-CP,South Korea,resid cooling modern_d3,gas,2030,gas,1.0
698,None Specified,Other-CP,South Korea,resid cooling modern_d4,gas,2030,gas,1.0
766,None Specified,Other-CP,South Korea,resid cooling modern_d5,gas,2030,gas,1.0
834,None Specified,Other-CP,South Korea,resid cooling modern_d6,gas,2030,gas,1.0
902,None Specified,Other-CP,South Korea,resid cooling modern_d7,gas,2030,gas,1.0
970,None Specified,Other-CP,South Korea,resid cooling modern_d8,gas,2030,gas,1.0
1038,None Specified,Other-CP,South Korea,resid cooling modern_d9,gas,2030,gas,1.0


In [68]:
df = run_query(conn=conn, q_idx=122, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df[(df['Year'] >= 2025) & (df['Year'] <= 2030)]

iron and steel prices


,Units,scenario,region,sector,Year,value
6,1975$/kg,Other-CP,South Korea,iron and steel,2025,0.165651
7,1975$/kg,Other-CP,South Korea,iron and steel,2030,0.164223
28,1975$/kg,Other-EA,South Korea,iron and steel,2025,0.165653
29,1975$/kg,Other-EA,South Korea,iron and steel,2030,0.185811


In [69]:
0.156661 * 0.13 * 0.5

0.010182965

In [7]:
dfCO2 = run_query(conn=conn, q_idx=262, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
dfCO2['GHG'] = 'CO2'
dfNonCO2 = run_query(conn=conn, q_idx=270, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df = pd.concat([dfCO2, dfNonCO2])
df["emiss(MT)"] = df.apply(convert_to_mt, axis=1)  # uses your utils.convert_to_mt
df["gwpAr5"]    = df["GHG"].map(GWP_AR5).astype(float)
df["MTCO2eq"]   = df["emiss(MT)"] * df["gwpAr5"]
df[(~df['sector'].isin(['trn_aviation_intl', 'trn_shipping_intl']))].groupby(['scenario', 'Year'])['MTCO2eq'].sum()

CO2 emissions by sector (no bio) (excluding resource production)
nonCO2 emissions by sector (excluding resource production)


scenario  Year
Other-CP  1975     54.590644
          1990    295.171616
          2005    588.834536
          2010    694.980605
          2015    750.755896
          2020    702.605474
          2025    689.204194
          2030    636.915075
          2035    558.711188
Other-EA  1975     54.590644
          1990    295.171616
          2005    588.834536
          2010    694.980605
          2015    750.755896
          2020    718.322268
          2025    692.298845
          2030    531.951948
          2035    360.789351
Name: MTCO2eq, dtype: float64

In [10]:
dfCO2 = run_query(conn=conn, q_idx=265, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
dfCO2['GHG'] = 'CO2'

CO2 emissions by tech (excluding resource production)


In [11]:
dfCO2[(dfCO2['Year'] >= 2035) & (dfCO2['sector'].str.contains('resid'))]

,Units,scenario,region,sector,subsector,technology,Year,value,GHG
764,MTC,Other-CP,South Korea,resid heating coal_d1,coal,coal,2035,0.038586,CO2
776,MTC,Other-CP,South Korea,resid heating coal_d2,coal,coal,2035,0.029975,CO2
784,MTC,Other-CP,South Korea,resid heating coal_d3,coal,coal,2035,0.026305,CO2
792,MTC,Other-CP,South Korea,resid heating coal_d4,coal,coal,2035,0.013154,CO2
831,MTC,Other-CP,South Korea,resid heating modern_d1,gas,gas,2035,0.228668,CO2
...,...,...,...,...,...,...,...,...,...
2505,MTC,Other-EA,South Korea,resid others modern_d5,gas,gas,2035,0.208632,CO2
2520,MTC,Other-EA,South Korea,resid others modern_d6,gas,gas,2035,0.219550,CO2
2535,MTC,Other-EA,South Korea,resid others modern_d7,gas,gas,2035,0.231642,CO2
2550,MTC,Other-EA,South Korea,resid others modern_d8,gas,gas,2035,0.246399,CO2


In [9]:
558.711188 - 38.3

520.411188

In [10]:
1 - 520.411188 / 742.3

0.29892066819345264

In [32]:
dfCO2Map   = pd.read_csv("./extdata/gcamreport/CO2_tech_map.csv", skiprows=[0])
dfNonCO2Map= pd.read_csv("./extdata/gcamreport/nonCO2_emissions_sector_map.csv", skiprows=[0])

In [32]:
df = run_query(conn=conn, q_idx=265, scenarios=['Other-CP','Other-EA'], regions=['South Korea'])
df['sector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

CO2 emissions by tech (excluding resource production)


array(['H2 central production', 'H2 wholesale dispensing',
       'agricultural energy use', 'airCO2', 'ammonia',
       'backup_electricity', 'cement', 'chemical energy use',
       'chemical feedstocks', 'comm cooling', 'comm heating',
       'comm others', 'construction energy use', 'desalinated water',
       'elec_biomass (IGCC)', 'elec_biomass (conv)',
       'elec_coal (IGCC CCS)', 'elec_coal (conv pul CCS)',
       'elec_coal (conv pul)', 'elec_gas (CC CCS)', 'elec_gas (CC)',
       'elec_gas (steam/CT)', 'elec_refined liquids (CC)',
       'elec_refined liquids (steam/CT)', 'electricity', 'gas processing',
       'iron and steel', 'mining energy use',
       'other industrial energy use', 'process heat cement',
       'process heat dac', 'process heat food processing',
       'process heat paper', 'refining', 'regional biomass',
       'regional biomassOil', 'regional corn for ethanol',
       'regional woodpulp for energy', 'resid heating coal_d1',
       'resid heating coal_

In [8]:
df = run_query(conn=conn, q_idx=273, scenarios=['Power-CP'], regions=['South Korea'])
df['sector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

nonCO2 emissions by tech (excluding resource production)


array(['comm cooling', 'electricity_net_ownuse', 'industrial processes',
       'resid cooling modern_d1', 'resid cooling modern_d10',
       'resid cooling modern_d2', 'resid cooling modern_d3',
       'resid cooling modern_d4', 'resid cooling modern_d5',
       'resid cooling modern_d6', 'resid cooling modern_d7',
       'resid cooling modern_d8', 'resid cooling modern_d9',
       'urban processes', 'Beef', 'Corn', 'Dairy', 'FiberCrop',
       'FodderGrass', 'Fruits', 'H2 central production',
       'H2 liquid truck', 'H2 pipeline', 'H2 wholesale dispensing',
       'Legumes', 'MiscCrop', 'NutsSeeds', 'OilCrop', 'OtherGrain',
       'Pork', 'Poultry', 'Rice', 'RootTuber', 'SheepGoat', 'Soybean',
       'UnmanagedLand', 'Vegetables', 'Wheat', 'agricultural energy use',
       'ammonia', 'backup_electricity', 'biomass', 'chemical energy use',
       'comm heating', 'comm others', 'construction energy use',
       'electricity', 'iron and steel', 'mining energy use',
       'other indus

In [9]:
listCrop = [
    'Corn', 'FiberCrop', 'FodderGrass', 'Fruits', 'Legumes', 'MiscCrop', 'NutsSeeds', 
    'OilCrop', 'OtherGrain', 'RootTuber', 'Soybean', 'Vegetables', 'Wheat',
]

In [10]:
df[(df['sector'].isin(listCrop))]['technology'].unique()

array(['CornC4_Korea_RFD_hi', 'CornC4_Korea_RFD_lo',
       'FiberCrop_Korea_IRR_hi', 'FiberCrop_Korea_IRR_lo',
       'FiberCrop_Korea_RFD_hi', 'FiberCrop_Korea_RFD_lo',
       'FodderGrass_Korea_RFD_hi', 'FodderGrass_Korea_RFD_lo',
       'FruitsTree_Korea_IRR_hi', 'FruitsTree_Korea_IRR_lo',
       'FruitsTree_Korea_RFD_hi', 'FruitsTree_Korea_RFD_lo',
       'Fruits_Korea_IRR_hi', 'Fruits_Korea_IRR_lo',
       'Fruits_Korea_RFD_hi', 'Fruits_Korea_RFD_lo',
       'Legumes_Korea_RFD_hi', 'Legumes_Korea_RFD_lo',
       'MiscCrop_Korea_IRR_hi', 'MiscCrop_Korea_IRR_lo',
       'MiscCrop_Korea_RFD_hi', 'MiscCrop_Korea_RFD_lo',
       'NutsSeedsTree_Korea_IRR_hi', 'NutsSeedsTree_Korea_IRR_lo',
       'NutsSeedsTree_Korea_RFD_hi', 'NutsSeedsTree_Korea_RFD_lo',
       'NutsSeeds_Korea_RFD_hi', 'NutsSeeds_Korea_RFD_lo',
       'OilCrop_Korea_IRR_hi', 'OilCrop_Korea_IRR_lo',
       'OilCrop_Korea_RFD_hi', 'OilCrop_Korea_RFD_lo',
       'OtherGrainC4_Korea_RFD_hi', 'OtherGrainC4_Korea_RFD_lo',
 

In [11]:
df["emiss(MT)"] = df.apply(convert_to_mt, axis=1)  # uses your utils.convert_to_mt
df["gwpAr5"]    = df["GHG"].map(GWP_AR5).astype(float)
df["MTCO2eq"]   = df["emiss(MT)"] * df["gwpAr5"]
df[(df['sector'].isin(listCrop))].groupby(['Year', 'scenario'])['MTCO2eq'].sum()

Year  scenario
1975  Power-CP    1.956932
1990  Power-CP    2.296362
2005  Power-CP    2.277849
2010  Power-CP    1.803642
2015  Power-CP    2.001942
2020  Power-CP    2.081022
2025  Power-CP    2.119450
2030  Power-CP    2.152947
2035  Power-CP    2.178591
Name: MTCO2eq, dtype: float64

In [17]:
df[(df['GHG'].str.contains('HFC'))].groupby(['Year', 'scenario'])['MTCO2eq'].sum()

Year  scenario
1990  Power-CP     0.040795
2005  Power-CP    15.341120
2010  Power-CP    23.340671
2015  Power-CP    25.690444
2020  Power-CP    28.499158
2025  Power-CP    30.077220
2030  Power-CP    30.820361
2035  Power-CP    30.305728
Name: MTCO2eq, dtype: float64

In [ ]:
df = run_query(conn=conn, q_idx=166, scenarios=['Power-CP'], regions=['South Korea'])
df['subsector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

In [ ]:
df = run_query(conn=conn, q_idx=166, scenarios=['Power-CP'], regions=['South Korea'])
df['subsector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

In [17]:
df = run_query(conn=conn, q_idx=166, scenarios=['Power-CP'], regions=['South Korea'])
df['subsector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

costs of transport techs


array(['International Aviation', 'Domestic Aviation', 'HSR',
       'Passenger Rail', 'road', 'Bus', 'LDV', '2W and 3W', '4W', 'Car',
       'Large Car and Truck', 'Domestic Ship', 'Freight Rail',
       'Medium truck', 'International Ship'], dtype=object)

In [18]:
dfPol = df[((df['subsector'] == 'Domestic Ship')) & (df['Year'] >= 2025) & (df['Year'] <= 2035)].copy()
# dfPol['subsidy_cp'] = dfPol['value'] * (-0.2)
# dfPol['subsidy_cp'] = dfPol['value'] * (-0.2)
dfPol

,Units,scenario,region,sector,subsector,technology,Year,value
267,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,BEV,2025,0.015547
268,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,BEV,2030,0.011832
269,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,BEV,2035,0.008128
276,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,FCEV,2025,0.010209
277,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,FCEV,2030,0.009685
278,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,FCEV,2035,0.009247
285,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Hybrid Liquids,2025,0.005660
286,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Hybrid Liquids,2030,0.005644
287,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Hybrid Liquids,2035,0.005389
294,1990$/ton-km,Power-CP,South Korea,trn_freight,Domestic Ship,Liquids,2025,0.006466


In [7]:
df = run_query(conn=conn, q_idx=170, scenarios=['Power-CP'], regions=['South Korea'])
df['sector'].unique()#[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

transport tech share-weights


array(['trn_aviation_intl', 'trn_freight', 'trn_freight_road', 'trn_pass',
       'trn_pass_road', 'trn_pass_road_LDV', 'trn_pass_road_LDV_4W',
       'trn_shipping_intl'], dtype=object)

In [11]:
df[(df['subsector'] == 'Medium truck') & (df['Year'] >= 2020) & (df['Year'] <= 2035)]

,Units,scenario,region,sector,subsector,Year,technology,value
237,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Hybrid Liquids,0.158869
238,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Liquids,1.000000
239,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,BEV,0.034445
240,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,FCEV,0.034445
241,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Hybrid Liquids,0.841131
242,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Liquids,1.000000
243,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,BEV,0.158869
244,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,FCEV,0.158869
245,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Hybrid Liquids,1.000000
246,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Liquids,1.000000


In [10]:
df[(df['sector'] == 'trn_freight_road') & (df['Year'] >= 2020) & (df['Year'] <= 2035)]

,Units,scenario,region,sector,subsector,Year,technology,value
237,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Hybrid Liquids,0.158869
238,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2020,Liquids,1.000000
239,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,BEV,0.034445
240,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,FCEV,0.034445
241,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Hybrid Liquids,0.841131
242,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2025,Liquids,1.000000
243,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,BEV,0.158869
244,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,FCEV,0.158869
245,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Hybrid Liquids,1.000000
246,None Specified,Power-CP,South Korea,trn_freight_road,Medium truck,2030,Liquids,1.000000


In [9]:
df[(df['sector'] == 'trn_pass_road_LDV_4W') & (df['Year'] >= 2020) & (df['Year'] <= 2035)]

,Units,scenario,region,sector,subsector,Year,technology,value
666,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2020,Hybrid Liquids,0.158869
667,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2020,Liquids,1.000000
668,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2020,NG,0.001004
669,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,BEV,0.500000
670,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,FCEV,0.500000
671,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,Hybrid Liquids,0.841131
672,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,Liquids,1.000000
673,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2025,NG,0.001004
674,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2030,BEV,1.000000
675,None Specified,Power-CP,South Korea,trn_pass_road_LDV_4W,Car,2030,FCEV,1.000000


In [24]:
df = run_query(conn=conn, q_idx=100, scenarios=['Power-CP'], regions=['South Korea'])
df[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

industry final energy by tech and fuel


,Units,scenario,region,sector,subsector,technology,input,Year,value
83,EJ,Power-CP,South Korea,chemical energy use,biomass,biomass,delivered biomass,2030,0.011847
87,EJ,Power-CP,South Korea,chemical energy use,biomass,biomass CCS,delivered biomass,2030,0.000059
95,EJ,Power-CP,South Korea,chemical energy use,coal,coal,delivered coal,2030,0.004963
99,EJ,Power-CP,South Korea,chemical energy use,coal,coal CCS,delivered coal,2030,0.000010
108,EJ,Power-CP,South Korea,chemical energy use,electricity,electricity,elect_td_ind,2030,0.189090
114,EJ,Power-CP,South Korea,chemical energy use,gas,gas,wholesale gas,2030,0.048086
118,EJ,Power-CP,South Korea,chemical energy use,gas,gas CCS,wholesale gas,2030,0.000561
126,EJ,Power-CP,South Korea,chemical energy use,refined liquids,refined liquids,refined liquids industrial,2030,0.022880
130,EJ,Power-CP,South Korea,chemical energy use,refined liquids,refined liquids CCS,refined liquids industrial,2030,0.000352
136,EJ,Power-CP,South Korea,chemical feedstocks,coal,coal,delivered coal,2030,0.024484


In [23]:
df = run_query(conn=conn, q_idx=268, scenarios=['Power-CP'], regions=['South Korea'])
df[(df['Year'] == 2030) & (df['sector'].str.contains('chemical'))]#['value'].sum()

CO2 sequestration by tech


,Units,scenario,region,sector,subsector,technology,Year,value
16,MTC,Power-CP,South Korea,chemical energy use,biomass,biomass CCS,2030,0.001231
20,MTC,Power-CP,South Korea,chemical energy use,coal,coal CCS,2030,0.000237
24,MTC,Power-CP,South Korea,chemical energy use,gas,gas CCS,2030,0.007169
28,MTC,Power-CP,South Korea,chemical energy use,refined liquids,refined liquids CCS,2030,0.006206
37,MTC,Power-CP,South Korea,chemical feedstocks,refined liquids,refined liquids,2030,36.714500


In [13]:
df = run_query(conn=conn, q_idx=321, scenarios=['Power-CP'], regions=['South Korea'])
df#[(df['Year'] == 2020) & (df['subsector'] == 'gas')]#['value'].sum()

costs by tech


,Units,scenario,region,sector,subsector,Year,technology,value
0,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,1975,woodpulp_energy,6.923830
1,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,1990,woodpulp_energy,98.457800
2,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,2005,woodpulp_energy,51.642100
3,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,2010,woodpulp_energy,45.543000
4,$/GJ,Power-CP,South Korea,woodpulp_energy,woodpulp_energy,2015,woodpulp_energy,26.855700
...,...,...,...,...,...,...,...,...
6175,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2030,Liquids,0.002578
6176,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2035,BEV,0.004874
6177,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2035,FCEV,0.004116
6178,1990$/ton-km,Power-CP,South Korea,trn_shipping_intl,International Ship,2035,Hybrid Liquids,0.002262


In [14]:
df[(df['sector'] == 'carbon-storage')]

,Units,scenario,region,sector,subsector,Year,technology,value
5982,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,1975,offshore carbon-storage,212.00
5983,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,1990,offshore carbon-storage,212.00
5984,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2005,offshore carbon-storage,212.00
5985,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2010,offshore carbon-storage,212.00
5986,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2015,offshore carbon-storage,212.00
5987,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2020,offshore carbon-storage,212.00
5988,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2025,offshore carbon-storage,212.00
5989,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2030,offshore carbon-storage,212.00
5990,1990$/tC,Power-CP,South Korea,carbon-storage,offshore carbon-storage,2035,offshore carbon-storage,212.00
5991,1990$/tC,Power-CP,South Korea,carbon-storage,onshore carbon-storage,1975,onshore carbon-storage,1.00


In [25]:
df = run_query(conn=conn, q_idx=11, scenarios=['Power-CP'], regions=['South Korea'])
df[(df['Year'] == 2020) & (df['subsector'] == 'gas')]#['value'].sum()

elec gen by gen tech and cooling tech and vintage


,Units,scenario,region,sector,subsector,technology,output,Year,value
224,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (dry cooling),year=2015",elec_gas (CC),2020,0.012519
228,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (dry cooling),year=2020",elec_gas (CC),2020,0.001774
242,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (once through),year=2015",elec_gas (CC),2020,0.002087
250,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (recirculating),year=2015",elec_gas (CC),2020,0.270339
254,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (recirculating),year=2020",elec_gas (CC),2020,0.067489
268,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (seawater),year=2015",elec_gas (CC),2020,0.068611
272,EJ,Power-CP,South Korea,electricity,gas,"gas (CC) (seawater),year=2020",elec_gas (CC),2020,0.018227
286,EJ,Power-CP,South Korea,electricity,gas,"gas (steam/CT) (dry cooling),year=2015",elec_gas (steam/CT),2020,0.000078
290,EJ,Power-CP,South Korea,electricity,gas,"gas (steam/CT) (dry cooling),year=2020",elec_gas (steam/CT),2020,0.000016
304,EJ,Power-CP,South Korea,electricity,gas,"gas (steam/CT) (once through),year=2015",elec_gas (steam/CT),2020,0.000010


In [17]:
5260 / 4333

1.2139395338102932

In [18]:
5920 / 4333

1.3662589429956151

In [20]:
5530 / 4333

1.2762520193861067

In [22]:
3090 / 4333

0.7131317793676437